_This notebook was developed by [Keneth Garcia](https://www.linkedin.com/in/keneth-garcia-a6305b1b9/). Source and license info are on [GitHub](https://github.com/KenethGarcia/SieveComP)._

# Python API Quickstart

This notebook introduces the core Python API of `SieveComP`. We will walk through setting up the engine, executing a TOML configuration, and interacting with the resulting data using Pandas.

By the end of this quickstart, you will know how to load a configuration, run the quality-control pipeline, inspect the output DataFrames, and rapidly iterate on your rules without re-querying the database.

# 1. Setting up the connection to the database

SieveComP can connect to multiple databases using either a configuration file, an env file or environment variables. For this tutorial, we will use an env file named `.env` in our working directory with the following content:

```env
# Credentials for SeisComP6 database
SERVER_SC6_USER=consulta
SERVER_SC6_PASSWORD=consulta
SERVER_SC6_HOST=10.100.100.xxx
SERVER_SC6_PORT=3306
SERVER_SC6_DATABASE=seiscomp
```

However, feel free to use your own credentials in any of the supported formats. The engine will automatically detect and load the credentials.

# 2. Setting the SQL query

SieveComP uses SQL queries to extract data from the database. These queries are stored in separate `.sql` files, which are referenced in the TOML configuration file. For this tutorial, we will use a very simple query that retrieves the last 20 events from the `Event` table in the SeisComP database. The query is stored in a file named `../../examples/data/sql/example.sql` with the following content:

```sql
SELECT * FROM Event LIMIT 20;
```

## 3. The Configuration File

SieveComP is driven by human-readable TOML files. For this tutorial, assume we have a file named `tutorial_config.toml` in our working directory with the following content:

```toml
[[database]]
name = "connection1"
env_file = "/home/keneth/Documents/Academic_Projects/SieveComP/examples/data/credentials/.env"
env_prefix = "SERVER_SC6"

[[queries]]
name        = "query1"
sql_file    = "/home/keneth/Documents/Academic_Projects/SieveComP/examples/data/sql/example.sql"
description = "Query 1 — Get the last 20 events from the Event table"

[[checks]]
name = "Check events with a methodID"

  [[checks.conditions]]
  rule_type = "category"
  column = "methodID"
  mode = "in"
  values = ["Hypo71"]

[output]
columns = ["_oid", "time_value", "methodID"]
```

This file defines a database connection, a SQL query, a duplicate detection method, and a quality check to retrieve only events with a specific methodID. If you need help to create your own TOML configuration, please refer to the [TOML Configuration Guide](https://github.com/KenethGarcia/SieveComP/blob/1c9e22709beb15882b9a3361db2e634fa0c34299/docs/TOML_schema.md).

> NOTE: You can modify these file to suit your needs, all these files will be stored at the examples/data directory of the SieveComP repository. You can also create your own files and point to them in the TOML configuration.

## 4. Initializing the Runner

The `Runner` class is the central orchestator. When you instantiate it, SieveComP reads your TOML file, rigorously validates your schema, and establishes the required database connections:



In [1]:
import pandas as pd
from sievecomp.core.runner import Runner

# Initialize the Runner with your configuration file
runner = Runner("../../examples/data/configs/tutorial_config.toml")

# You can verify which queries and checks were successfully loaded:
print("Active Queries:", runner.list_queries())
print("Active Checks:", runner.list_checks())

Active Queries: ['query1']
Active Checks: ['Check events with a methodID']


## 5. Executing the Pipeline

The `run()` method executes the entire quality-control pipeline. It fetches the events via SQL, runs the duplicate detection algorithms (Adjacent or SSWA), evaluates all spatial/logical checks, and generates the final merged output table.

In [2]:
# Execute the pipeline
result = runner.run()

/tmp/ipykernel_57957/98149272.py:2: UserWarning: No [[polygons]] entries found in the configuration. Spatial checks will not be available.
  result = runner.run()


## 6. Inspecting the Output

The `run()` method returns a `RunResult` data class. This object contains four distinct Pandas DataFrames, allowing you to inspect the data at various stages of the pipeline.

### 6.1. Total Events

`result.total_events` contains the raw, unmodified dataset pulled directly from your SeisComP database.

In [3]:
print(f"Total events fetched: {len(result.total_events)}")
result.total_events

Total events fetched: 20


,_oid,_parent_oid,_last_modified,time_value,time_value_ms,time_uncertainty,time_lowerUncertainty,time_upperUncertainty,time_confidenceLevel,time_pdf_variable_content,...,creationInfo_agencyID,creationInfo_agencyURI,creationInfo_author,creationInfo_authorURI,creationInfo_creationTime,creationInfo_creationTime_ms,creationInfo_modificationTime,creationInfo_modificationTime_ms,creationInfo_version,creationInfo_used
0,213701015,1,2026-10-08 18:22:50,2026-10-08 05:41:29+00:00,797368,0.408789,None,None,None,None,...,SGC,,gerard@proc3,,2026-10-08 18:22:06,632110,2026-10-08 18:22:50,557764.0,,1
1,213694366,1,2026-10-08 17:53:56,2026-10-08 09:39:54+00:00,0,NaN,None,None,None,None,...,SGC,,gerard@proc3,,2026-10-08 17:53:55,657323,2026-10-08 17:53:56,617086.0,,1
2,213695098,1,2026-10-08 17:58:32,2026-10-08 09:46:21+00:00,920000,NaN,None,None,None,None,...,SGC,,gerard@proc3,,2026-10-08 17:58:17,543946,2026-10-08 17:58:32,867767.0,,1
3,213694531,1,2026-10-08 17:55:31,2026-10-08 09:52:55+00:00,0,NaN,None,None,None,None,...,SGC,,gerard@proc3,,2026-10-08 17:55:30,442603,2026-10-08 17:55:31,594110.0,,1
4,213695886,1,2026-10-08 18:01:59,2026-10-08 09:59:16+00:00,609999,NaN,None,None,None,None,...,SGC,,gerard@proc3,,2026-10-08 18:01:40,506395,2026-10-08 18:01:59,613468.0,,1
5,213696082,1,2026-10-08 18:02:18,2026-10-08 09:59:17+00:00,35417,NaN,None,None,None,None,...,SGC,,gerard@proc3,,2026-10-08 18:02:04,666025,2026-10-08 18:02:18,317213.0,,1
6,213696670,1,2026-10-08 18:06:06,2026-10-08 10:51:36+00:00,0,NaN,None,None,None,None,...,SGC,,muruena@proc2,,2026-10-08 18:06:06,46445,2026-10-08 18:06:06,991147.0,,1
7,213696790,1,2026-10-08 18:06:53,2026-10-08 10:53:55+00:00,0,NaN,None,None,None,None,...,SGC,,muruena@proc2,,2026-10-08 18:06:52,790717,2026-10-08 18:06:53,719627.0,,1
8,213697004,1,2026-10-08 18:07:26,2026-10-08 10:55:35+00:00,0,NaN,None,None,None,None,...,SGC,,muruena@proc2,,2026-10-08 18:07:24,695617,2026-10-08 18:07:26,71465.0,,1
9,213697481,1,2026-10-08 18:08:15,2026-10-08 10:58:12+00:00,0,NaN,None,None,None,None,...,SGC,,muruena@proc2,,2026-10-08 18:08:14,504317,2026-10-08 18:08:15,487864.0,,1


### 6.2. Duplicate & Checks

`result.duplicates` contains the events flagged as duplicates by the selected algorithm, while `result.checks` contains the results of all spatial/logical checks. They feature an appended `Observations` column detailing exactly which rules were broken.

In [4]:
print(f"Potential duplicates found: {len(result.duplicates)}")
print(f"Events satisfying quality checks: {len(result.checks)}")

result.checks

Potential duplicates found: 0
Events satisfying quality checks: 2


,_oid,_parent_oid,_last_modified,time_value,time_value_ms,time_uncertainty,time_lowerUncertainty,time_upperUncertainty,time_confidenceLevel,time_pdf_variable_content,...,creationInfo_agencyURI,creationInfo_author,creationInfo_authorURI,creationInfo_creationTime,creationInfo_creationTime_ms,creationInfo_modificationTime,creationInfo_modificationTime_ms,creationInfo_version,creationInfo_used,Observations
2,213695098,1,2026-10-08 17:58:32,2026-10-08 09:46:21+00:00,920000,NaN,None,None,None,None,...,,gerard@proc3,,2026-10-08 17:58:17,543946,2026-10-08 17:58:32,867767.0,,1,Check events with a methodID
4,213695886,1,2026-10-08 18:01:59,2026-10-08 09:59:16+00:00,609999,NaN,None,None,None,None,...,,gerard@proc3,,2026-10-08 18:01:40,506395,2026-10-08 18:01:59,613468.0,,1,Check events with a methodID


### 6.3. Final Output

`result.output` is the compiled review table. If you defined an `[output]` section in your TOML, this DataFrame will strictly follow your column selections and format preferences.

In [5]:
# View the final formatted table
result.output

,_oid,time_value,methodID,Observations
0,213695098,2026-10-08 09:46:21+00:00,Hypo71,Check events with a methodID
1,213695886,2026-10-08 09:59:16+00:00,Hypo71,Check events with a methodID


## 7. Rapid Iteration (Using the Cache)

Designing the perfect quality-control rules often requires trial and error. If you decide to tweak a parameter in your TOML file (e.g., changing the category filter to a different value), you do not need to wait for the database query to run again.

Use `update_from_cache()` to instantly apply your new TOML rules to the data you already downloaded.

In [8]:
# 1. Edit 'tutorial_config.toml' in your text editor and save it.
# 2. Re-run the engine using the cached data:

new_result = runner.update_from_cache(
    events_df=result.total_events,
    reload_config=True  # Updated by changing check mode from in to not_in in the TOML file
)

print(f"Updated flagged events based on new rules: {len(new_result.checks)}")
new_result.output

Updated flagged events based on new rules: 18


/tmp/ipykernel_57957/1699081627.py:4: UserWarning: No [[polygons]] entries found in the configuration. Spatial checks will not be available.
  new_result = runner.update_from_cache(


,_oid,time_value,methodID,Observations
0,213701015,2026-10-08 05:41:29+00:00,LOCSAT,Check events without Hypo71 methodID
1,213694366,2026-10-08 09:39:54+00:00,,Check events without Hypo71 methodID
2,213694531,2026-10-08 09:52:55+00:00,,Check events without Hypo71 methodID
3,213696082,2026-10-08 09:59:17+00:00,NonLinLoc,Check events without Hypo71 methodID
4,213696670,2026-10-08 10:51:36+00:00,,Check events without Hypo71 methodID
5,213696790,2026-10-08 10:53:55+00:00,,Check events without Hypo71 methodID
6,213697004,2026-10-08 10:55:35+00:00,,Check events without Hypo71 methodID
7,213697481,2026-10-08 10:58:12+00:00,,Check events without Hypo71 methodID
8,213695392,2026-10-08 17:56:30+00:00,LOCSAT,Check events without Hypo71 methodID
9,213698156,2026-10-08 18:06:49+00:00,LOCSAT,Check events without Hypo71 methodID
